# Calculating Clean and Dirty Prices With FEDInvest Data
<br>
<br>


<div style="display: flex; flex-wrap: wrap; align-items: center; gap: 15px; margin-bottom: 25px; padding-bottom: 15px; border-bottom: 1px solid #eaeaea;">
  
  <a href="https://colab.research.google.com/github/PatrickJHess/Volume-Three-Chapter-Two/blob/master/colab/Colab_FEDInvest_accrued_interest_Clean_Dirty_Prices.ipynb" target="_blank" style="display: flex; align-items: center;">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 28px; margin: 0;">
  </a>

  <a href="https://mybinder.org/v2/gh/PatrickJHess/Volume-Three-Chapter-Two/master?urlpath=lab/tree/notebooks/FEDInvest_accrued_interest_Clean_Dirty_Prices.ipynb" target="_blank" style="background-color: #f5a252; color: white; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">🚀</span> Launch Live in Binder
  </a>

  <a href="https://patrickjhess.github.io/Volume-Three-Chapter-Two/" style="background-color: #f1f3f4; color: #3c4043; border: 1px solid #dadce0; padding: 0 12px; text-decoration: none; font-weight: bold; border-radius: 4px; font-family: sans-serif; display: flex; align-items: center; font-size: 0.9em; height: 28px; box-sizing: border-box;">
    <span style="margin-right: 6px; font-size: 1.1em;">⬅️</span> Return to Main Book
  </a>

</div>

## A Begining Thought

Before diving into the specific code, we want you to notice how brief this main script is. In just a few lines, we are going to pull raw data from the web, clean it up, and run complex financial calculations across an entire portfolio of bonds.

On the surface, it looks almost entirely effortless. As we walk through these three steps and the `apply` method, keep that simplicity in mind. Afterward, we'll talk about the 'iceberg principle' happening behind the scenes that makes Python so incredibly powerful for this kind of work.

## Using the new functions to create a DataFrame of bond prices
This workflow utilizes three main functions to extract, clean, and process bond data:

* **`FEDInvest`**: Scrapes the raw data directly from the FEDInvest page on the Treasury Direct website.

* **`Clean_FEDInvest`**: Processes the raw data returned by `FEDInvest` to create a structured, cleaned DataFrame.

* **`accrued_interest`**: Calculates the accrued interest for an individual bond based on its specific parameters.

* **Streamlining Calculations with `apply`**:To efficiently calculate the accrued interest for every bond in our dataset, we introduce the pandas `apply` method. Instead of using a slow, manual loop, `apply` executes the `accrued_interest` function across each row of the cleaned DataFrame simultaneously. This approach seamlessly computes the accrued interest for each bond and stores the result as a new column in the DataFrame.

:::{important} [ ▼ ] How to use this page: Run, Copy, & Download
:class: dropdown

<ul>
  <li><b>⏻ Run code right here:</b> Click the <b>Power Button</b> icon at the top of the screen to activate <b>Live Code</b>.</li>
  <li><b>📋 Copy code:</b> Hover over any code block and click the <b>Clipboard icon</b> in the top-right corner.</li>
  <li><b>📥 Download this file:</b> Click the <b>Download icon</b> (downward arrow) at the top right of the screen to save this exact notebook to your computer.</li>
</ul>
:::

:::{important} 🛠️ Notebook Setup: Why the "Try/Except" Imports?
:class: dropdown

**The Goal:**
To ensure this notebook runs perfectly whether you are using **Google Colab**, a local **Jupyter instance**, or a remote server without you having to manually install software.

* **External Libraries:** NumPy and Pandas are the "heavy hitters" for data. They aren't always installed by default.
* **The `try/except` Logic:** This is a safety net.
    1. We **try** to import the library.
    2. If it fails (because it's not installed), the **except** block triggers a `!pip install` to download it automatically.
* **Aliasing (`as np`):** We rename `numpy` to `np` to save keystrokes. In professional finance code, `np` and `pd` are the universal shorthand.
  
:::

## Preparing the notebook

### Importing libraries, modules, And functions

Modules that are included in the standard Python library are imported. When necessary, other modules or libraries are installed before they are imported. The library and module (python-dateutil and Calendar) are introduced in the *Mainipulating Dates* notebook of this chapter (see [Control Statements](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/Control_Statements.html#the-try-and-except)).

```
import os
import sys
import requests
from datetime import , date
from types import ModuleType
import calendar

try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd
```

In [1]:
import os
import sys
import requests
from datetime import date, datetime
from types import ModuleType
import calendar

try:
    import numpy as np
except:
    !pip -q install numpy
    import numpy as np

try:
    import pandas as pd
except:
    !pip -q install pandas
    import pandas as pd

:::{important} ☁️ Cloud-Loading: How In-Memory Modules Work
:class: dropdown

**The Logic:**
Usually, Python looks for modules as `.py` files on your hard drive. Here, we are "tricking" Python into treating a string of text from a URL as a live library.

**The Workflow:**
1. **Fetch:** `requests.get(url)` grabs the raw text of your Python script from Dropbox.
2. **Instantiate:** `ModuleType(module_name)` creates an empty "container" in your computer's RAM.
3. **Execute:** `exec(code, module.__dict__)` runs that text inside the container, turning text into live functions.
4. **Register:** By adding it to `sys.modules`, we tell Python: *"If I try to import this later, don't look on the disk—look right here in the memory."*

**Why do this?**
It makes your notebooks **100% portable**. A user can open this in a brand-new environment, and as long as they have an internet connection, all your custom financial functions will "just work."
:::

### Adding a custom module and importing functions


Similar to Chapter One, this notebook utilizes the custom module, **module_basic_concepts_fixed_income**, sourced from Dropbox and named `basic_concepts_fixed_income`. As a reminder, the module is accessible in the notebook's memory, but is not added to a drive.  

The functions `FEDInvest` and `clean_FEDInvest` of the notebook *Treasury Direct Data* are imported. In addition two new functions (`scheduled_bond_payments` and `accrued_interest`) are imported.


```
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from module_basic_concepts_fixed_income import (FEDInvest,
                                                   clean_FEDInvest,
                                                   accrued_interest)
                                                   
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
    # Now we can import from our in-memory module
```



In [2]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
    response = requests.get(url)
    module = ModuleType(module_name)
    exec(response.text, module.__dict__)
    sys.modules[module_name] = module
    # Now we can import from our in-memory module
    from basic_concepts_fixed_income import (FEDInvest,
                                             clean_FEDInvest,
                                             accrued_interest)
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
    # Now we can import from our in-memory module

## Using `FEDInvest` and `clean_FEDInvest` to access and create a clean DataFrame

* **`FEDInvest`** princing date required.
* **`clean_FEDInvest`** DataFrame is required
* **`stamp`** & **`settlement`** are the pricing and settlement dates


````{dropdown} 🔍 Click to see the <code>FEDInvest</code> function

```python
def FEDInvest(price_date):
  """
    Fetches historical security prices from the FedInvest portal.

    Args:
        price_date (datetime.date): The date for which to retrieve prices.
            Note: Current day is typically available after 1:00 PM ET on business days.


    Returns:
        tuple: (pandas.DataFrame, str) if successful. The DataFrame contains
               security details (CUSIP, Price, Yield), and the string is the
               official "Prices For" date stamp from the site.
        tuple: (str, None) if the request fails or no data is found for the date
                (attempt to fetch current day before 1:00 PM ET).

    Example:
        >>> from datetime import date
        >>> df, stamp = FEDInvest(date(2025, 3, 17))
  """
  import requests
  from io import StringIO
  import pandas as pd
  from datetime import datetime, date
  from dateutil.relativedelta import relativedelta

  # check for date or datetime
  validate_date(price_date)

  # make share date of prices and settlement date are settlement dates
  price_date=adjust_bond_pay_dates(price_date)
  if price_date > date.today():
    return "price_date is in the future", None, None
  
  settlement_date=price_date+relativedelta(days=1)
  settlement_date=adjust_bond_pay_dates(settlement_date)

  # URL address of Treasury Direct Select A Date
  url = "https://treasurydirect.gov/GA-FI/FedInvest/selectSecurityPriceDate"

  # Standard headers to look like a real browser
  headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36\
     (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Content-Type": "application/x-www-form-urlencoded"
  }

  #  variable names and type identified from inspecting url
  month=str(price_date.month)
  day=str(price_date.day)
  year=str(price_date.year)

  # payload passed in request post
  payload={'priceDate.month':month,
           'priceDate.day':day,
           'priceDate.year':year,
           "submit": "Show Prices"}

  # fires off form and returns prices for date
  try:
        response = requests.post(url, data=payload, headers=headers, timeout=10)
        response.raise_for_status()
  except requests.exceptions.RequestException as e:
        return f"Connection Error: {e}", None

  # reads the html
  # Pandas recommends to wrap the response in StingIO to make file like
  tables=pd.read_html(StringIO(response.text),match='CUSIP')

  # from inspection there is a single table
  return tables[0], price_date,settlement_date


```
````

````{dropdown} 🔍  Click to see the <code>clean_FEDInvest</code> function

```python
def clean_FEDInvest(df):

    import pandas as pd
    # Filters for Standard Securities
    keep_rows=df['SECURITY TYPE'].str.contains('bill|note|bond',case=False)
    security_df=df[keep_rows].copy()

    # Removes Clutter
    drop_columns=['CUSIP','CALL DATE']
    security_df.drop(columns=drop_columns,inplace=True)

    # Creates a Time-Series Index
    security_df.set_index('MATURITY DATE',inplace=True)
    security_df.index=pd.to_datetime(security_df.index)
    security_df.sort_index(inplace=True)

    # Standardizes Financial Terms
    change_column_names={'RATE':'Coupon',
                         'BUY':'Price Ask',
                         'SELL':'Price Bid'}
    security_df.rename(columns=change_column_names,inplace=True)

    # Formats Numeric Data
    numeric_cols = ['Coupon', 'Price Ask', 'Price Bid', 'YIELD']
    for col in numeric_cols:
        if col in security_df.columns:
            security_df[col] = security_df[col].astype(str).str.replace('%', '', regex=False).astype(float)

    return security_df
```
````


## Scrap and clean the data with `FEDInvest` and `clean_FEDInvest`

The `tail` method is used to display the last five rows of the cleaned Dataframe.

In [4]:
from datetime import datetime
my_date=datetime(2026, 3, 17)

# Fetch Treasury data for a specific date
df, stamp,settlement = FEDInvest(my_date)

# not refernced here, but notr trade date (stamp) and settlement are assigned
# clean the data
clean_df=clean_FEDInvest(df)
display(clean_df.tail())

,SECURITY TYPE,Coupon,Price Ask,Price Bid,END OF DAY
MATURITY DATE,,,,,
2055-02-15,MARKET BASED BOND,4.625,96.390625,96.37500,96.31250
2055-05-15,MARKET BASED BOND,4.750,98.390625,98.37500,98.31250
2055-08-15,MARKET BASED BOND,4.750,98.453125,98.43750,98.40625
2055-11-15,MARKET BASED BOND,4.625,96.484375,96.46875,96.43750
2056-02-15,MARKET BASED BOND,4.750,98.515625,98.50000,98.43750


## Use Panda's vectorized column methods to calculate the `'Clean Price'` column

In standard mark-to-market applications, the bid/ask average (mid-price) is used as the clean price. In the raw FEDInvest data, however, there are instances where the bid or ask quote is missing and defaults to zero. To account for this without skewing our valuations, the clean price is calculated using a maximum function between the bid/ask average and the closing price (`'End Of Day'`). Because a missing quote results in an artificially low or zero-value average, this method effectively falls back to the closing price whenever bid/ask data is incomplete.

The `head` method displays the first five rows and the `shape` attribute the total number of rows and columns.

In [ ]:
clean_df['Clean Price']= np.maximum((clean_df['Price Ask']+clean_df['Price Bid'])/2,
                             clean_df['END OF DAY'])
display(clean_df.head(),clean_df.shape)

,SECURITY TYPE,Coupon,Price Ask,Price Bid,END OF DAY,Clean Price
MATURITY DATE,,,,,,
2026-03-19,MARKET BASED BILL,0.00,0.000000,99.980222,99.990111,99.990111
2026-03-24,MARKET BASED BILL,0.00,99.929806,99.929611,99.940000,99.940000
2026-03-26,MARKET BASED BILL,0.00,99.910000,99.909750,99.919778,99.919778
2026-03-31,MARKET BASED NOTE,4.50,0.000000,100.031250,100.031250,100.031250
2026-03-31,MARKET BASED NOTE,0.75,0.000000,99.875000,99.906250,99.906250


(399, 6)

## Using the `apply` method to add accrued interest to the `clean_df` (399 bills, notes, & bonds) DataFrame.

Although Pandas excels at fast, vectorized column operations (such as calculating clean prices), standard vectorization falls short when dealing with complex, conditional row-by-row logic. This is where the `apply` method becomes essential (see [Pandas method apply](https://patrickjhess.github.io/Introduction-To-Python-For-Financial-Python/An_Introduction_To_Pandas.html#the-apply-method)).

Calculating accrued interest across a mixed portfolio is a perfect example of where simple column-wise math is not feasible. The `apply` method solves this by passing data directly to a custom function.

The primary benefits of using `apply` include:

1. **Row-Level Flexibility:** A lambda function can directly access and evaluate multiple specific columns simultaneously within a single row.  
2. **Simplified Code:** It completely eliminates the need to write slow, clunky `for` loops, leading to cleaner and more maintainable code.

Before we calculate the accrued interest for the 399 securities in `clean_df`, let's demonstrate the mechanics of the method using a simplified DataFrame called `rows_to_columns`. The calculations of this example *could* be solved with basic column operations, its simplicity makes the underlying behavior of `apply` much easier to understand.

````{dropdown} ✂️ Copy the snippet to demonstrate the apply method


```python
import pandas as pd
import numpy as np

# create the DataFrame
rows_to_columns = pd.DataFrame({
    'Rates': [0.03, 0.05, 0.07],   # future & present values assumes continuous
    'Future Values': [1.030455, 1.05171, 1.072508],
    'Present Value': [0.970446, 0.951229, 0.93294]
})
display(rows_to_columns)

# demonstrate the apply method
# assume rates are discrete and convert to continuous
rows_to_columns['Continuous'] = rows_to_columns.apply(
    lambda row: np.log(1 + row['Rates']),  # row is the name assigned to each row...could be a differnt name
    axis=1      # axis 0 for columns and 1 fr rows        
)
display(rows_to_columns)
```
````

## Using the `apply` method to calculate accrued intrest

````{dropdown} 🔍 Click to see <code>accrued_intrest</code>

```py
def accrued_interest(maturity, coupon, day_type='Actual/Actual', settlement=None, freq=2):
    """
      Returns the accrued interest for a bond.  Returns the accrued interest for a bond.

    Args:
        maturity (datetime): The maturity date of the bond.
        coupon (float): The annual coupon rate (e.g., 0.05 for 5%).
        day_types:
            Actual/Actual, Actual/365, Actual/360. and 30/360.
        settlement (datetime, optional): The settlement date. Defaults to today.
        freq (int, optional):
        Coupon frequency per year: 1 (annual). 2 (semi-annual)
                                    4 (quarterly), 12 (monthly)
                                    Defaults to 2.
    """
    from datetime import datetime,date
    import calendar
    from dateutil.relativedelta import relativedelta
    #Validate Data
    def validate_date(datetime_object):
      # check for datetime or date
      if not isinstance(datetime_object, (datetime, date)):
          raise TypeError("Input must be a datetime or date object.")
      # convert datetime to date
      if isinstance(datetime_object, datetime):
        datetime_object = datetime_object.date()
      return datetime_object
    maturity = validate_date(maturity)

    if settlement is None:
        settlement = date.today()
    else:
        settlement = validate_date(settlement)

    if freq not in [1, 2, 4, 12]:
        print(f"⚠️ Warning: Freq {freq} invalid. Assumed Semi-Annual (2).")
        freq = 2

    try:
        coupon = float(coupon)
        if coupon < 0: raise ValueError
    except:
        raise ValueError("Coupon must be a positive number.")

    # Define strategic_date
    # Get all the bond's potential payment dates in the next year
    mat_is_last=maturity.day==calendar.monthrange(maturity.year,maturity.month)[1]
    if mat_is_last:
      lastDay=calendar.monthrange(settlement.year+1,maturity.month)[1]
      strategic_date=date(settlement.year+1,maturity.month,lastDay)
    else:
     strategic_date=date(settlement.year+1,maturity.month,maturity.day)

    #The strategic date: minimum of actual and next year's maturity date
    strategic_date=min(maturity,strategic_date)
    pay_dates = scheduled_pay_dates(strategic_date, settlement=settlement, freq=freq)

    # Should sorted but check
    pay_dates.sort()

    # The first date after the settlement date is the next coupon date
    next_coupon = None
    for d in pay_dates:
        if d >= settlement:
            next_coupon = d
            break

    # Bond has matured or annual coupon is zero
    if next_coupon is None or coupon==0:
        return 0.0

    #Calculate Previous Coupon Date
    num_months = int(12 // freq)
    prev_coupon = next_coupon - relativedelta(months=num_months)

    # Check for Month End adjustment on the calculated previous date
    is_next_month_end = next_coupon.day == calendar.monthrange(maturity.year, maturity.month)[1]

    if is_next_month_end:
        last_day_of_prev_month = calendar.monthrange(prev_coupon.year, prev_coupon.month)[1]
        prev_coupon = date(prev_coupon.year, prev_coupon.month, last_day_of_prev_month)

    # The day a coupon is paid is also the first day of the new cycle.

    accrued_value = 0.0

    if day_type == 'Actual/Actual':
        days_since_last = (settlement - prev_coupon).days
        days_between = (next_coupon - prev_coupon).days
        #  (DaysHeld / DaysInPeriod)
        accural_ratio= days_since_last/days_between
        # Actual/Actual uses the coupon paid on the date
        accrued_value = (coupon / freq) * accural_ratio


    elif day_type == '30/360':
        accural_ratio =_30_360_(settlement,prev_coupon)
        # Formula: Coupon * accrural ratio
        accrued_value = coupon * accural_ratio

    elif day_type == 'Actual/360':
        days_since_last = (settlement - prev_coupon).days
        accural_ratio= days_since_last/360
      # Formula: Coupon * accrural ratio        
        accrued_value = coupon * accural_ratio

    elif day_type == 'Actual/365':
        accural_ratio = convert_isda(settlement,prev_coupon)
      # Formula: Coupon * accrural ratio        
        accrued_value = coupon * accural_ratio

    else:
        # Fallback
        print(f"⚠️ Warning: Unknown day_type {day_type}. Using Actual/Actual.")
        days_since_last = (settlement - prev_coupon).days
        days_between = (next_coupon - prev_coupon).days
        #  (DaysHeld / DaysInPeriod)
        accural_ratio= days_since_last/days_between
        # Actual/Actual uses the coupon paid on the date
        accrued_value = (coupon / freq) * (days_since_last / days_between)

    return accrued_value
 ```
 ```

In [ ]:
# Settlement date returned by FEDInvest
clean_df['Accrued Interest'] = clean_df.apply(
    lambda row: accrued_interest(
        maturity=row.name,        # Value of index is the maturity date
        coupon=row['Coupon'],
        settlement=settlement
    ),
    axis=1                        # Explicitly state axis=1 for readability
)

# Vectorize the dirty price calculation
clean_df['Dirty Price'] = clean_df['Clean Price'] + clean_df['Accrued Interest']

display(clean_df)

,SECURITY TYPE,Coupon,Price Ask,Price Bid,END OF DAY,Clean Price,Accrued Interest,Dirty Price
MATURITY DATE,,,,,,,,
2026-03-19,MARKET BASED BILL,0.000,0.000000,99.980222,99.990111,99.990111,0.000000,99.990111
2026-03-24,MARKET BASED BILL,0.000,99.929806,99.929611,99.940000,99.940000,0.000000,99.940000
2026-03-26,MARKET BASED BILL,0.000,99.910000,99.909750,99.919778,99.919778,0.000000,99.919778
2026-03-31,MARKET BASED NOTE,4.500,0.000000,100.031250,100.031250,100.031250,2.089286,102.120536
2026-03-31,MARKET BASED NOTE,0.750,0.000000,99.875000,99.906250,99.906250,0.348214,100.254464
...,...,...,...,...,...,...,...,...
2055-02-15,MARKET BASED BOND,4.625,96.390625,96.375000,96.312500,96.382812,0.396064,96.778876
2055-05-15,MARKET BASED BOND,4.750,98.390625,98.375000,98.312500,98.382812,1.613950,99.996763
2055-08-15,MARKET BASED BOND,4.750,98.453125,98.437500,98.406250,98.445312,0.406768,98.852080


### **The Power of Abstraction: Python's Hidden Heavy Lifting**

This workflow is a perfect illustration of why Python is so widely used in data science and finance: the power of modular abstraction. In programming, abstraction is the concept of hiding complex, behind-the-scenes work so the user can focus solely on the high-level task at hand.

Think of this script like an iceberg.

**The Tip of the Iceberg: The User's Code** From the perspective of the main script, the code is elegantly simple. With just a few lines of code and the use of pandas' `.apply()` method, we can trigger an entire pipeline that fetches, cleans, and calculates data for a whole portfolio of bonds. The user doesn't need to write the complex web-scraping logic or the intricate financial math formulas in their main workspace.

**Below the Surface: The Custom Module** The real heavy lifting happens entirely out of sight within the imported custom module. When we call a function like `FEDInvest` or `accrued_interest`, we are kicking off a massive chain reaction.

* These functions rely on internal helper functions hidden inside the module.  
* The module itself seamlessly handles its own dependencies, importing third-party libraries (like `requests` for scraping or `numpy` for math) exactly when and where they are needed.

**Why This Matters** By encapsulating all this complexity into a custom module, Python allows us to accomplish sophisticated, multi-step tasks with minimal effort. This modular approach provides three massive benefits:

1. **Readability:** The main script remains clean, uncluttered, and easy for anyone to read and understand.  
2. **Reusability:** The web scraping and financial calculations are written once in the module, meaning they can be imported and used across dozens of different projects without rewriting a single line of code.  
3. **Efficiency:** The user gets to act as a high-level manager, simply delegating the hard work to the modules and libraries, letting Python do exactly what it was designed to do: maximize output while minimizing user effort.

:::{dropdown} ✨ AI Study Assistant
:open:

Select a prompt below to open Google AI and explore these concepts further. *(Sign in to save your chat history!)*

---

✨ [**Is accrued interest of a bond related to its risk of default? ↗**](https://www.google.com/search?udm=50&q=Is+accrued+interest+of+a+bond+related+to+its+risk+of+default? )

---

✨ [**Does selling a bond a few days before a coupon payment avoid taxes on the coupon? ↗**](https://www.google.com/search?udm=50&q=Does+selling+a+bond+a+few+days+before+a+coupon+payment+avoid+taxes+on+the+coupon?)

---

✨ [**The 30/360 rule of corporate bonds should result in less accrued interest than the actual/actual rule of Treasury bonds? ↗**](https://www.google.com/search?udm=50&q=The+30/360+rule+of+corporate+bonds+should+result+in+less+accrued+interest+than+the+actual/actual+rule+of+Treasury+bonds?)

---

✨ [**The clean price of bonds is always less than the dirty price? ↗**](https://www.google.com/search?udm=50&q=The+clean+price+of+bonds+is+always+less+than+the+dirty+price?)
:::